# VoxPhysica 3 cm Speaker-Level Research Pipeline

This notebook runs the direct speaker-level height challenger. It is separate from the clip-level neural trainer: one row per speaker, rich pooled acoustic statistics, a validation-tuned ensemble, and explicit short/tall slice metrics.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

FEATURES_DIR = ROOT / 'data' / 'features_audited'
OUTPUT_DIR = ROOT / 'outputs' / 'research_height_ensemble' / 'seed_11'
SEED = 11
TARGET_MAE_CM = 3.0

print('root:', ROOT)
print('features:', FEATURES_DIR)
print('output:', OUTPUT_DIR)

In [ ]:
required = [FEATURES_DIR / split for split in ('train', 'val', 'test')]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing audited feature splits. Rebuild or restore data/features_audited before running this notebook. '
        f'Missing: {missing}'
    )

for split in ('train', 'val', 'test'):
    files = list((FEATURES_DIR / split).glob('*.npz'))
    print(f'{split}: {len(files)} clips')

In [ ]:
from src.research.speaker_height_ensemble import load_research_tables

tables = load_research_tables(str(FEATURES_DIR))
for split, table in tables.items():
    print(split, table.x.shape, 'height range', (float(table.y.min()), float(table.y.max())))
print('feature count:', len(tables['train'].feature_names))

In [ ]:
from src.research.speaker_height_ensemble import run_research_experiment

payload = run_research_experiment(
    features_dir=str(FEATURES_DIR),
    output_dir=str(OUTPUT_DIR),
    seed=SEED,
    target_mae_cm=TARGET_MAE_CM,
    ensemble_trials=5000,
)

print('target met:', payload['target_met'])
print('val MAE:', payload['final_val']['calibrated_edge']['mae'])
print('test MAE:', payload['final_test']['calibrated_edge']['mae'])

In [ ]:
import pandas as pd

metrics = payload['final_test']['calibrated_edge']
summary = pd.DataFrame(
    [
        {'slice': 'overall', 'mae_cm': metrics['mae'], 'n': metrics['n_speakers']},
        {'slice': 'short', 'mae_cm': metrics.get('short_mae'), 'n': metrics.get('short_n')},
        {'slice': 'medium', 'mae_cm': metrics.get('medium_mae'), 'n': metrics.get('medium_n')},
        {'slice': 'tall', 'mae_cm': metrics.get('tall_mae'), 'n': metrics.get('tall_n')},
        {'slice': 'edge_mean', 'mae_cm': metrics.get('edge_mae'), 'n': None},
        {'slice': 'edge_worst', 'mae_cm': metrics.get('edge_worst_mae'), 'n': None},
    ]
)
summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

preds = pd.read_csv(OUTPUT_DIR / 'predictions_test.csv')
plt.figure(figsize=(7, 5))
sns.scatterplot(data=preds, x='height_cm', y='calibrated_edge_pred_cm', hue='height_bin', s=70)
lo = min(preds['height_cm'].min(), preds['calibrated_edge_pred_cm'].min()) - 2
hi = max(preds['height_cm'].max(), preds['calibrated_edge_pred_cm'].max()) + 2
plt.plot([lo, hi], [lo, hi], color='black', linewidth=1)
plt.xlabel('True height (cm)')
plt.ylabel('Predicted height (cm)')
plt.title('Held-out test speakers')
plt.show()

In [ ]:
weights = pd.Series(payload['ensemble_weights']).sort_values(ascending=False)
display(weights.to_frame('weight'))

with open(OUTPUT_DIR / 'summary.md', 'r', encoding='utf-8') as handle:
    print(handle.read())